# Tutorial: The MLflow MCP Server

![The MLflow MCP Server](images/mlflow_mcp_server.svg)

## Turn MLflow into a set of tools your AI assistant can call

The **Model Context Protocol (MCP)** is an open standard that lets AI assistants (Claude, VS Code, Cursor,
your own agents) call external *tools* through a uniform interface. 

The **MLflow MCP Server** —
`mlflow mcp run` — exposes MLflow operations (searching traces, reading a trace, logging feedback, managing
experiments/runs/models) as MCP tools. Point an assistant at it and it can *observe and act on your MLflow
data for you*: for instance, "find the failed traces from the last hour", "show me the slowest ones", "log a relevance
score on this trace."

### What You'll Learn
- Start the MLflow MCP Server and connect a programmatic MCP client to it.
- Control which tools are exposed with `MLFLOW_MCP_TOOLS`.
- Use the built-in tools for four real workflows: **debug** failures, **analyze** latency, **log** quality
  feedback, and **clean up** data — with `extract_fields` to keep responses small.
- Wire the server into Claude, VS Code, and Cursor.

### Prerequisites
- The `mlflow[mcp]` extra installed (already in this repo's `pyproject.toml`; brings in `fastmcp`).
- **A running MLflow tracking server:**
  ```bash
  uv run mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
  ```

> **Note:** The MLflow MCP Server is **experimental** (requires MLflow ≥ 3.5.1) and its API may evolve.
> The companion tutorial, [The MLflow MCP Registry](./mlflow_mcp_registry.ipynb), covers *cataloging* MCP servers.

### Estimated Time: 10–15 minutes

---
## Step 1: Setup & verify

Point MLflow at your running tracking server, pick an experiment, and confirm `fastmcp` is available.

In [1]:
import os
import sys
import mlflow
from dotenv import load_dotenv
from packaging.version import Version
import fastmcp  # provided by the mlflow[mcp] extra


load_dotenv()

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI  # the MCP server subprocess reads this from the env
mlflow.set_tracking_uri(TRACKING_URI)

EXPERIMENT_NAME = "mcp-server-demo"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

assert Version(mlflow.__version__) >= Version("3.5.1"), (
    f"The MLflow MCP Server needs MLflow >= 3.5.1 (found {mlflow.__version__})."
)
print(f"✅ MLflow version:   {mlflow.__version__}")
print(f"✅ fastmcp version:  {fastmcp.__version__}")
print(f"✅ Tracking URI:     {TRACKING_URI}")
print(f"✅ Experiment:       {experiment.name} (id={experiment.experiment_id})")

2026/08/28 11:10:02 INFO mlflow.tracking.fluent: Experiment with name 'mcp-server-demo' does not exist. Creating a new experiment.


✅ MLflow version:   3.15.1
✅ fastmcp version:  3.4.7
✅ Tracking URI:     http://localhost:5000
✅ Experiment:       mcp-server-demo (id=1)


---
## Step 2: Seed some traces to work with

So the tools have realistic data to act on, generate a handful of traces an assistant would triage. We use
plain `@mlflow.trace` functions (no LLM or API key needed); notebook
[03](./03_introduction_to_tracing.ipynb) shows `mlflow.openai.autolog()` for real LLM calls.

We tag each function with a **`span_type`** so the traces are labeled by the kind of work they represent

The set below is chosen to exercise the whole tracing surface, so the tools in Step 4 have something rich to
query:

| Function | Span type(s) | Showcases |
|----------|-------------|-----------|
| `answer` | `LLM` | a healthy (OK) trace; we log feedback on one later |
| `slow_report` | `TASK` | high **latency** |
| `flaky_tool` | `TOOL` | an **ERROR** trace |
| `rag_answer` | `CHAIN` → `RETRIEVER` + `LLM` | a **multi-span hierarchy** (parent + typed children) |
| `production_query` | `LLM` | custom **tags** for metadata filtering |
| `chat_with_usage` | `CHAT_MODEL` | **token usage** (cost observability) |

`mlflow.flush_trace_async_logging()` forces the async trace exporter to finish so the traces are immediately
queryable.

In [2]:
import time
from mlflow.entities import SpanType


@mlflow.trace(span_type=SpanType.LLM)
def answer(question: str) -> str:
    """Stand in for an LLM answering a user question.

    Purpose: produce a healthy (OK) trace tagged as an LLM span — the kind an
    assistant call would create. We attach human feedback to one of these later.
    Returns a canned string so the notebook runs with no API key.
    """
    return f"Here is a helpful answer to: {question}"


@mlflow.trace(span_type=SpanType.TASK)
def slow_report() -> str:
    """Simulate a slow downstream task.

    Purpose: produce an OK but deliberately slow (~1.3s) trace so the "analyze
    latency" search (execution_time_ms > 1000) has a slow trace to return.
    """
    time.sleep(1.3)  # simulate a slow downstream call
    return "generated a large report"


@mlflow.trace(span_type=SpanType.TOOL)
def flaky_tool() -> str:
    """A tool call that fails on purpose.

    Purpose: raise an exception so MLflow records the trace with ERROR status,
    giving the "debug failures" search (status = 'ERROR') a real failure to find.
    """
    raise RuntimeError("simulated downstream failure")


@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve(question: str) -> list[str]:
    """Retrieve context documents for a question (a RETRIEVER span).

    Purpose: a child step of the RAG trace below. RETRIEVER is the span type the UI
    and RAG judges (RetrievalGroundedness, RetrievalRelevance) key off of.
    """
    return ["doc: MLflow Tracing records spans", "doc: spans nest into a trace"]


@mlflow.trace(span_type=SpanType.LLM)
def generate(question: str, docs: list[str]) -> str:
    """Generate an answer from retrieved context (an LLM span, child of the RAG trace)."""
    return f"Based on {len(docs)} documents: here is the answer to '{question}'."


@mlflow.trace(span_type=SpanType.CHAIN)
def rag_answer(question: str) -> str:
    """A small RAG pipeline: retrieve, then generate.

    Purpose: produce a single trace with a **multi-span hierarchy** — a CHAIN parent
    with RETRIEVER and LLM children — so `get_trace` can reveal the structure a real
    agent/RAG app produces.
    """
    docs = retrieve(question)
    return generate(question, docs)


@mlflow.trace(span_type=SpanType.LLM)
def production_query(question: str) -> str:
    """Answer a question and tag the trace with production metadata.

    Purpose: attach custom **tags** (via `update_current_trace`) so we can filter by
    them — e.g. `search_traces(filter="tags.environment = 'production'")` — the "which
    traces came from prod / a premium user" workflow.
    """
    mlflow.update_current_trace(tags={"environment": "production", "user_tier": "premium"})
    return f"Production answer to: {question}"


@mlflow.trace(span_type=SpanType.CHAT_MODEL)
def chat_with_usage(question: str) -> str:
    """A chat call that records token usage on its span.

    Purpose: showcase **cost/usage observability**. Setting `mlflow.chat.tokenUsage`
    on the span rolls up to `trace.info.token_usage`, so an assistant can report token
    spend. (Auto-tracing captures this automatically for real providers.)
    """
    span = mlflow.get_current_active_span()
    span.set_attribute(
        "mlflow.chat.tokenUsage",
        {"input_tokens": 42, "output_tokens": 58, "total_tokens": 100},
    )
    return f"Chat answer to: {question}"


# A healthy trace we'll attach feedback to later.
answer("What is MLflow Tracing?")
mlflow.flush_trace_async_logging()

# save the trace id for later use
ok_trace_id = mlflow.get_last_active_trace_id()

# More OK/slow/error traces.
answer("How do I evaluate an agent?")
slow_report()
try:
    flaky_tool()
except RuntimeError:
    pass  # the failure is captured in the trace as an ERROR

# A nested RAG trace — capture its id to inspect the span hierarchy in Step 4.
rag_answer("How does MLflow tracing work?")
mlflow.flush_trace_async_logging()

# save the trace id for later use
rag_trace_id = mlflow.get_last_active_trace_id()

# A tagged production trace and a chat trace with token usage.
production_query("What is my order status?")
chat_with_usage("Summarize the MCP docs.")
mlflow.flush_trace_async_logging()
chat_trace_id = mlflow.get_last_active_trace_id()

print(f"✅ Seeded 7 traces in experiment {experiment.experiment_id}")
print(f"   Healthy trace to annotate later: {ok_trace_id}")
print(f"   Nested RAG trace to inspect:     {rag_trace_id}")
print(f"   Chat trace with token usage:     {chat_trace_id}")

✅ Seeded 7 traces in experiment 1
   Healthy trace to annotate later: tr-20a098b0831ab60e82f04c3d256f4ac8
   Nested RAG trace to inspect:     tr-2db16e9eae7518035eb395a54c0935e5
   Chat trace with token usage:     tr-8d7d2b1cefa99a3e2691342c2fb6db82


[Trace(trace_id=tr-20a098b0831ab60e82f04c3d256f4ac8), Trace(trace_id=tr-400f54b0dd8e3fa260b78206603cc6bd), Trace(trace_id=tr-fbeb4f4250da6b2a5e8cbfdbc887b7c9), Trace(trace_id=tr-97743b6ce1ed2e8f50456298347077db), Trace(trace_id=tr-2db16e9eae7518035eb395a54c0935e5), Trace(trace_id=tr-63c42832d0f15034c4d6113993fc1e50), Trace(trace_id=tr-8d7d2b1cefa99a3e2691342c2fb6db82)]

---
## Step 3: Connect an MCP client and discover the tools

`mlflow mcp run` speaks MCP over **stdio**. We launch it as a subprocess and connect with `fastmcp.Client`.

> **Under the hood — what "over stdio" means.** The client launches `mlflow mcp run` as a **subprocess** and
> the two exchange **JSON-RPC messages over its standard I/O pipes**: requests go to the child's `stdin`,
> responses come back on its `stdout`, and `stderr` is left for logs. It's plain pipe-based IPC — no network
> socket — so the server is spawned on demand and lives only as long as the client. (The custom and external
> servers in the [Registry tutorial](./mlflow_mcp_registry.ipynb) instead use the `streamable-http` transport,
> which is JSON-RPC over HTTP, for long-running or remote servers.)

The environment variable **`MLFLOW_MCP_TOOLS`** controls which tool *categories* are exposed (smaller tool
lists mean less token overhead for an assistant):

| `MLFLOW_MCP_TOOLS` | Categories | Tool count |
|--------------------|-----------|-----------|
| `genai` *(default)* | traces, scorers, experiments, runs | 26 |
| `ml` | experiments, runs, models, deployments | 32 |
| `all` | everything above | 45 |
| comma-separated | a custom subset, e.g. `"traces,scorers"` | varies |

Below we start the server with the default `genai` toolset and list what it exposes. We also define a small
`mcp(...)` helper that opens a client, calls one tool, and returns its text — we'll reuse it throughout.

In [3]:
import asyncio
import nest_asyncio
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

# Jupyter already runs an event loop; nest_asyncio lets us use asyncio.run() inside it.
nest_asyncio.apply()


def mcp_transport(tools: str = "genai") -> StdioTransport:
    """Launch `mlflow mcp run` over stdio with a chosen MLFLOW_MCP_TOOLS set.

    sys.executable + "-m mlflow" keeps this portable (no dependency on `uv` being on PATH).
    """
    return StdioTransport(
        command=sys.executable,
        args=["-m", "mlflow", "mcp", "run"],
        env={**os.environ, "MLFLOW_MCP_TOOLS": tools},
    )


def mcp(tool: str, arguments: dict | None = None, tools: str = "genai") -> str:
    """Call one MLflow MCP tool and return its text output."""
    async def _call():
        async with Client(mcp_transport(tools)) as client:
            result = await client.call_tool(tool, arguments or {})
            return result.content[0].text

    return asyncio.run(_call())


async def _list_tools():
    async with Client(mcp_transport("genai")) as client:
        return await client.list_tools()


tools = asyncio.run(_list_tools())
print(f"🔧 The MLflow MCP Server exposes {len(tools)} tools with MLFLOW_MCP_TOOLS=genai:\n")
for t in tools:
    print(f"   - {t.name}")

🔧 The MLflow MCP Server exposes 26 tools with MLFLOW_MCP_TOOLS=genai:

   - search_traces
   - get_trace
   - delete_traces
   - set_trace_tag
   - delete_trace_tag
   - log_trace_feedback
   - log_trace_expectation
   - get_trace_assessment
   - update_trace_assessment
   - delete_trace_assessment
   - evaluate_traces
   - list_scorers
   - register_llm_judge_scorer
   - create_experiment
   - update_experiment
   - search_experiments
   - get_experiment
   - delete_experiment
   - restore_experiment
   - rename_experiment
   - list_runs
   - delete_run
   - restore_run
   - describe_run
   - create_run
   - link_traces_to_run


---
## Step 4: Use the tools

There are two ways to use the MCP tools from this MLflow MCP Server: 
 1. Use FastMCP Client
 2. Use AI Assistant

Each of the four workflows below is a single real tool call. In practice an assistant issues these for you
from natural language — the prompt it would receive is shown above each call.

### 4a. Debug failures

> *"Find the failed traces in my experiment."*

`search_traces` accepts a `filter_string` (same grammar as the MLflow UI) and an optional `extract_fields`
to return only the columns you care about. Its output is a compact table.

In [4]:
print(mcp("search_traces", {
    "experiment_id": experiment.experiment_id,
    "filter_string": "status = 'ERROR'",
    "extract_fields": "info.trace_id,info.state,info.request_preview",
}))

info.trace_id                        info.state    info.request_preview  
-----------------------------------  ------------  ----------------------
tr-97743b6ce1ed2e8f50456298347077db  ERROR         {}


### 4b. Analyze latency

> *"Show me the traces that took longer than a second."*

Filter on `execution_time_ms` and pull just the duration field.

In [5]:
print(mcp("search_traces", {
    "experiment_id": experiment.experiment_id,
    "filter_string": "execution_time_ms > 1000",
    "extract_fields": "info.trace_id,info.execution_duration_ms",
}))

info.trace_id                        info.execution_duration_ms  
-----------------------------------  ----------------------------
tr-fbeb4f4250da6b2a5e8cbfdbc887b7c9  1.3s


### 4c. Log quality feedback

> *"Log a relevance score of 0.9 on this trace, with a rationale."*

`log_trace_feedback` records an assessment on a trace (great for human review or LLM-judge scores). Note the
`value` is passed as a JSON-encoded string. We then read it back with `get_trace`, selecting only the
assessments — `get_trace` returns JSON.

In [6]:
feedback_result = mcp("log_trace_feedback", {
    "trace_id": ok_trace_id,
    "name": "relevance",
    "value": "0.9",              # JSON-encoded value
    "source_type": "HUMAN",
    "source_id": "reviewer@example.com",
    "rationale": "Answer is on-topic and accurate.",
})
print(feedback_result)

# Get the trace with the assessment afer setting 
# the relevance score to 0.9
print("\n📋 Assessments now on the trace:")
print(mcp("get_trace", {
    "trace_id": ok_trace_id,
    "extract_fields": "info.assessments.*",
}))

Logged feedback 'relevance' to trace tr-20a098b0831ab60e82f04c3d256f4ac8. Assessment ID: a-c59b06a248114ebea4dec0bcc1538ed8

📋 Assessments now on the trace:
{
  "info": {
    "assessments": [
      {
        "assessment_id": "a-c59b06a248114ebea4dec0bcc1538ed8",
        "assessment_name": "relevance",
        "trace_id": "tr-20a098b0831ab60e82f04c3d256f4ac8",
        "source": {
          "source_type": "HUMAN",
          "source_id": "reviewer@example.com"
        },
        "create_time": "2026-08-28T18:40:21.833Z",
        "last_update_time": "2026-08-28T18:40:21.833Z",
        "feedback": {
          "value": 0.9
        },
        "rationale": "Answer is on-topic and accurate.",
        "valid": true
      }
    ]
  }
}


### 4d. Beyond traces: experiments & runs

The `genai` toolset also manages **experiments** and **runs** (the `ml` set adds **models** and
**deployments**). For example, list the experiments on the server:

In [7]:
print(mcp("search_experiments", {"max_results": 5}))

Experiment Id    Name             Artifact Location  
---------------  ---------------  -------------------
0                Default          mlflow-artifacts:/0
1                mcp-server-demo  mlflow-artifacts:/1


### 4e. Rich traces: hierarchy, tags & token usage

The three richer traces we seeded show off more of what tracing captures — and each is one tool call away.

**Filter by tag** — find traces that came from production:

In [8]:
print(mcp("search_traces", {
    "experiment_id": experiment.experiment_id,
    "filter_string": "tags.environment = 'production'",
    "extract_fields": "info.trace_id,info.tags.environment,info.tags.user_tier",
}))

info.trace_id                        info.tags.environment    info.tags.user_tier  
-----------------------------------  -----------------------  ---------------------
tr-63c42832d0f15034c4d6113993fc1e50  production               premium


**Reveal the span hierarchy** of the RAG trace — `parent_span_id` links the RETRIEVER and LLM children
to their `CHAIN` parent (`rag_answer`, the root, has no parent):

In [9]:
print(mcp("get_trace", {
    "trace_id": rag_trace_id,
    "extract_fields": "data.spans.*.name,data.spans.*.span_id,data.spans.*.parent_span_id",
}))

{
  "data": {
    "spans": [
      {
        "name": "rag_answer",
        "span_id": "En8UKsv7FRc="
      },
      {
        "span_id": "tGCAjwUUXss=",
        "parent_span_id": "En8UKsv7FRc=",
        "name": "retrieve"
      },
      {
        "parent_span_id": "En8UKsv7FRc=",
        "name": "generate",
        "span_id": "Q3cJWZpxIvc="
      }
    ]
  }
}


**Read token usage** — the chat span records token counts under the standard `mlflow.chat.tokenUsage`
attribute (they also roll up to the trace's `token_usage`, shown in the UI). We fetch it with the same
`get_trace` tool by selecting the span attributes.

> Two notes: the field selector can return the whole `attributes` map but can't isolate the dotted
> `mlflow.chat.tokenUsage` key, so we grab `data.spans.*.attributes` and pick it out — and attribute values
> come back **JSON-encoded**, so we decode the one we want. (If you're in Python already, the SDK also exposes
> a tidy `mlflow.get_trace(id).info.token_usage` accessor.)

In [11]:
import json

detail = mcp("get_trace", {
    "trace_id": chat_trace_id,
    "extract_fields": "data.spans.*.attributes",
})

for span in json.loads(detail)["data"]["spans"]:
    raw_usage = span.get("attributes", {}).get("mlflow.chat.tokenUsage")
    if raw_usage:
        print(f"Token usage for the chat trace {chat_trace_id}:")
        print(f"   {json.loads(raw_usage)}")

Token usage for the chat trace tr-410147b79c55300a61f424fd029ab369:
   {'input_tokens': 42, 'output_tokens': 58, 'total_tokens': 100}


---
## Step 5: Wire the server into your assistant

You normally don't hand-write a client — you register the server once and its tools become available to your
assistant. Set `MLFLOW_TRACKING_URI` to your server (`http://localhost:5000`, a remote URL, or `databricks`
with `DATABRICKS_HOST` / `DATABRICKS_TOKEN`).

**Claude Code / Claude Desktop** (CLI, recommended):
```bash
claude mcp add mlflow-mcp -e MLFLOW_TRACKING_URI=http://localhost:5000 \
  -- uv run --with "mlflow[mcp]>=3.5.1" mlflow mcp run
```

**Claude — project-level `.mcp.json`:**
```json
{
  "mcpServers": {
    "mlflow-mcp": {
      "command": "uv",
      "args": ["run", "--with", "mlflow[mcp]>=3.5.1", "mlflow", "mcp", "run"],
      "env": { "MLFLOW_TRACKING_URI": "http://localhost:5000" }
    }
  }
}
```

**VS Code — `.vscode/mcp.json`** (note the `servers` key):
```json
{
  "servers": {
    "mlflow-mcp": {
      "command": "uv",
      "args": ["run", "--with", "mlflow[mcp]>=3.5.1", "mlflow", "mcp", "run"],
      "env": { "MLFLOW_TRACKING_URI": "http://localhost:5000" }
    }
  }
}
```

**Cursor — `.cursor/mcp.json`:**
```json
{
  "mcpServers": {
    "mlflow-mcp": {
      "command": "uv",
      "args": ["run", "--with", "mlflow[mcp]>=3.5.1", "mlflow", "mcp", "run"],
      "env": { "MLFLOW_TRACKING_URI": "http://localhost:5000", "MLFLOW_MCP_TOOLS": "genai" }
    }
  }
}
```

Then just ask, in natural language:
- *"Find all failed traces in experiment 1 from the last hour."*
- *"Show me the slowest traces with execution time over 5 seconds."*
- *"Log a relevance score of 0.85 for trace tr-… with a rationale about accuracy."*

---
## Step 6: Cleanup (optional)

Remove the traces we seeded. `delete_traces` accepts trace ids or a max timestamp; here we delete this
experiment's traces so the notebook stays re-runnable.

In [12]:
import time as _time

result = mcp("delete_traces", {
    "experiment_id": experiment.experiment_id,
    "max_timestamp_millis": int(_time.time() * 1000),
})
print(result)

Deleted 16 trace(s) from experiment 1.


---
## Summary

You ran the **MLflow MCP Server** and used it end to end:

- Started `mlflow mcp run` and connected a programmatic `fastmcp` client over stdio.
- Scoped the exposed tools with `MLFLOW_MCP_TOOLS` (genai / ml / all / custom subset).
- Used the built-in tools for four real workflows — **debug** (`search_traces status='ERROR'`),
  **latency** (`execution_time_ms`), **quality** (`log_trace_feedback` + `get_trace`), and **cleanup**
  (`delete_traces`) — with `extract_fields` to trim responses.
- Wired the server into Claude, VS Code, and Cursor.

### Next Steps
- **[The MLflow MCP Registry](./mlflow_mcp_registry.ipynb)** — catalog your own and third-party MCP servers,
  version them, and snapshot their tools.
- [07 — Evaluating Agents](./07_evaluating_agents.ipynb) — the feedback/assessment tools feed evaluation.
- MLflow docs: **MCP Server** — <https://mlflow.org/docs/latest/genai/mcp/>.